# Intro


The goal of this notebook is to experiment on neighbor-augmented predictions for pretrained time series forecasting models. We assume predictions and neighbors' data have already been extracted. We compute various new variables and models.


# Loading


## Imports


In [ ]:
from pathlib import Path

on_drive = True
if on_drive:

  from google.colab import drive
  drive.mount('/content/drive')
  data_path = "/content/drive/MyDrive/Recherche/Thèse Gaspard/Codes/FM adaptation/raw_payloads/electricity/678_10/"

else:
  data_path = ""

file_path = Path(data_path)

train_prediction_path = file_path / 'train_prediction_payload.pt'
eval_prediction_path = file_path / 'eval_prediction_payload.pt'
train_features_path = file_path / 'train_features_payload.pt'
eval_features_path = file_path / 'eval_features_payload.pt'

# Backward-compatible fallback, only if you manually point the old payloads here.
if not train_prediction_path.exists() and (file_path / 'train_payload.pt').exists():
    train_prediction_path = file_path / 'train_payload.pt'
if not eval_prediction_path.exists() and (file_path / 'eval_payload.pt').exists():
    eval_prediction_path = file_path / 'eval_payload.pt'

plots_dir = file_path / 'interactive_plots'
plots_dir.mkdir(parents=True, exist_ok=True)

print(f'Train prediction payload: {train_prediction_path.resolve()}')
print(f'Eval prediction payload:  {eval_prediction_path.resolve()}')
print(f'Train features payload:   {train_features_path.resolve()}')
print(f'Eval features payload:    {eval_features_path.resolve()}')
print(f'Plots dir:                {plots_dir.resolve()}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train prediction payload: /content/drive/MyDrive/Recherche/Thèse Gaspard/Codes/FM adaptation/raw_payloads/electricity/678_10/train_prediction_payload.pt
Eval prediction payload:  /content/drive/MyDrive/Recherche/Thèse Gaspard/Codes/FM adaptation/raw_payloads/electricity/678_10/eval_prediction_payload.pt
Train features payload:   /content/drive/MyDrive/Recherche/Thèse Gaspard/Codes/FM adaptation/raw_payloads/electricity/678_10/train_features_payload.pt
Eval features payload:    /content/drive/MyDrive/Recherche/Thèse Gaspard/Codes/FM adaptation/raw_payloads/electricity/678_10/eval_features_payload.pt
Plots dir:                /content/drive/MyDrive/Recherche/Thèse Gaspard/Codes/FM adaptation/raw_payloads/electricity/678_10/interactive_plots


In [ ]:
!pip install catboost

In [ ]:
from catboost import CatBoostClassifier

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import ipywidgets as widgets
from IPython.display import display, clear_output

from scipy.optimize import minimize_scalar
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## Load data


In [ ]:
def load_payload(path):
    if not path.exists():
        raise FileNotFoundError(path)
    return torch.load(path, map_location='cpu')


train_prediction_payload = load_payload(train_prediction_path)
eval_prediction_payload = load_payload(eval_prediction_path)
train_features_payload = load_payload(train_features_path)
eval_features_payload = load_payload(eval_features_path)

prediction_payloads = {'train': train_prediction_payload, 'eval': eval_prediction_payload}
features_payloads = {'train': train_features_payload, 'eval': eval_features_payload}


def get_tensor(payload, split, name):
    prefixed = f'{split}_{name}'
    if prefixed in payload:
        return payload[prefixed]
    if name in payload:
        return payload[name]
    raise KeyError(f'Missing {prefixed} or {name}. Available keys: {sorted(payload.keys())}')


def cat_prediction(name):
    return torch.cat(
        [get_tensor(prediction_payloads[split], split, name).float() for split in ['train', 'eval']],
        dim=0,
    )


def cat_feature(name):
    return torch.cat(
        [get_tensor(features_payloads[split], split, name).float() for split in ['train', 'eval']],
        dim=0,
    )


use_metadata_weights = False
metadata_time_penalty = 0.1
metadata_same_user_bonus = 0.25


def compute_retrieval_weights(payload, split, use_metadata=False):
    distance = get_tensor(payload, split, 'distance_x_xc').float()
    scores = -distance

    if use_metadata:
        query_t = get_tensor(payload, split, 'query_t').long().unsqueeze(-1)
        query_user = get_tensor(payload, split, 'query_user_idx').long().unsqueeze(-1)
        neighbor_t = get_tensor(payload, split, 'neighbor_t').long()
        neighbor_user = get_tensor(payload, split, 'neighbor_user_idx').long()
        period = max(int(get_tensor(payload, split, 'retrieval_period').item()), 1)

        delta_periods = (query_t - neighbor_t).clamp_min(0).float() / period
        same_user = (query_user == neighbor_user).float()
        scores = (
            scores
            - metadata_time_penalty * torch.log1p(delta_periods)
            + metadata_same_user_bonus * same_user
        )

    return torch.softmax(scores, dim=-1)


split_weights = {
    split: compute_retrieval_weights(
        prediction_payloads[split],
        split,
        use_metadata=use_metadata_weights,
    )
    for split in ['train', 'eval']
}


raw = {
    'y': cat_prediction('Y_values'),
    'yc': cat_prediction('Yc_values'),
    'pred': cat_prediction('preds'),
    'pred_c': cat_prediction('preds_context'),
    'e': cat_prediction('E_values'),
    'distance': cat_prediction('distance_x_xc'),
    'query_t': cat_prediction('query_t'),
    'query_user_idx': cat_prediction('query_user_idx'),
    'neighbor_t': cat_prediction('neighbor_t'),
    'neighbor_user_idx': cat_prediction('neighbor_user_idx'),
    'w': torch.cat([split_weights[split] for split in ['train', 'eval']], dim=0),
}


feature_raw = {}
for key, value in train_features_payload.items():
    if not torch.is_tensor(value):
        continue
    name = key.removeprefix('train_')
    try:
        feature_raw[name] = cat_feature(name)
    except KeyError:
        continue

query_t = raw['query_t'].unsqueeze(-1)
query_user_idx = raw['query_user_idx'].unsqueeze(-1)
neighbor_t = raw['neighbor_t']
neighbor_user_idx = raw['neighbor_user_idx']
retrieval_period = max(
    int(get_tensor(train_prediction_payload, 'train', 'retrieval_period').item()),
    1,
)
delta_periods = (query_t - neighbor_t).clamp_min(0).float() / retrieval_period
same_individual = (query_user_idx == neighbor_user_idx).float()

feature_raw['w_x_xc_entropy'] = -(raw['w'] * raw['w'].clamp_min(1e-8).log()).sum(dim=-1)
feature_raw['w_x_xc_max'] = raw['w'].max(dim=-1).values
feature_raw['w_x_xc_std'] = raw['w'].std(dim=-1, unbiased=False)
feature_raw['same_individual_mean'] = same_individual.mean(dim=-1)
feature_raw['delta_periods_mean'] = delta_periods.mean(dim=-1)
feature_raw['delta_periods_std'] = delta_periods.std(dim=-1, unbiased=False)

display(list(feature_raw.keys()))


## Computing features


In [ ]:
loss_fn = nn.MSELoss(reduction='none')
eps = 1e-8


def avg_horizon(x):
    return x.mean(dim=-1)


def weighted_neighbor_mean(x, w):
    return (w.unsqueeze(-1) * x).sum(dim=2)


def weighted_neighbor_horizon_loss(a, b, w):
    return (w.unsqueeze(-1) * (a - b).pow(2)).sum(dim=2).mean(dim=-1)


def pairwise_yc_loss(yc):
    if yc.shape[2] <= 1:
        return torch.zeros(yc.shape[:2], dtype=yc.dtype)
    diff = yc[:, :, :, None, :] - yc[:, :, None, :, :]
    return diff.pow(2).mean(dim=-1).mean(dim=(2, 3))


def flatten_window_feature(value):
    if torch.is_tensor(value):
        return value.detach().cpu().reshape(-1).numpy()
    return np.asarray(value).reshape(-1)


def make_feature_frame(feature_dict):
    return pd.DataFrame({name: flatten_window_feature(value) for name, value in feature_dict.items()})

y = raw['y']
yc = raw['yc']
pred = raw['pred']
pred_c = raw['pred_c']
e = raw['e']
w = raw['w']

ell_vanilla = loss_fn(pred, y)
ell_context = loss_fn(pred_c, y)
L = avg_horizon(ell_vanilla)
Lc = avg_horizon(ell_context)

neighbor_weighted_mean = weighted_neighbor_mean(yc, w)
neighbor_unweighted_mean = yc.mean(dim=2)
weighted_yc = neighbor_weighted_mean

weighted_e = weighted_neighbor_mean(e, w)

L_neighbor_weighted_mean = avg_horizon(loss_fn(neighbor_weighted_mean, y))
L_neighbor_unweighted_mean = avg_horizon(loss_fn(neighbor_unweighted_mean, y))
L_pred_plus_e = avg_horizon(loss_fn(pred + weighted_e, y))

L_oracle = torch.minimum(
    torch.minimum(torch.minimum(L, Lc), torch.minimum(L_neighbor_weighted_mean, L_neighbor_unweighted_mean)),
    L_pred_plus_e,
)

model_features = dict(feature_raw)
model_features.update({
    'loss_pred_pred_c': avg_horizon(loss_fn(pred, pred_c)),
    'loss_pred_neighbor_weighted_mean': avg_horizon(loss_fn(pred, neighbor_weighted_mean)),
    'loss_pred_neighbor_unweighted_mean': avg_horizon(loss_fn(pred, neighbor_unweighted_mean)),
    'loss_yc_pairwise': pairwise_yc_loss(yc),
    'weighted_e_norm': avg_horizon(weighted_e.pow(2)),
})

if 'mu_x' in model_features and 'mu_xc_mean' in model_features:
    model_features['mu_x_minus_mu_xc_mean'] = model_features['mu_x'] - model_features['mu_xc_mean']
if 'sigma_x' in model_features and 'sigma_xc_mean' in model_features:
    model_features['sigma_x_minus_sigma_xc_mean'] = model_features['sigma_x'] - model_features['sigma_xc_mean']

# Target-dependent quantities: use only for evaluation and plots, never as model inputs.
diagnostic_features = {
    'L_vanilla': L,
    'L_context': Lc,
    'L_neighbor_weighted_mean': L_neighbor_weighted_mean,
    'L_neighbor_unweighted_mean': L_neighbor_unweighted_mean,
    'L_pred_plus_weighted_e': L_pred_plus_e,
    'Imp_context': L - Lc,
    'Rel_Imp_context': 100.0 * (L - Lc) / L.clamp_min(eps),
    'Imp_neighbor_weighted_mean': L - L_neighbor_weighted_mean,
    'Rel_Imp_neighbor_weighted_mean': 100.0 * (L - L_neighbor_weighted_mean) / L.clamp_min(eps),
    'Imp_neighbor_unweighted_mean': L - L_neighbor_unweighted_mean,
    'Rel_Imp_neighbor_unweighted_mean': 100.0 * (L - L_neighbor_unweighted_mean) / L.clamp_min(eps),
    'Imp_pred_plus_weighted_e': L - L_pred_plus_e,
    'Rel_Imp_pred_plus_weighted_e': 100.0 * (L - L_pred_plus_e) / L.clamp_min(eps),
}

# Oracle diagnostics: also target-dependent
oracle_diagnostics = {
    'L_oracle': L_oracle,
    'Imp_oracle': L - L_oracle,
    'Rel_Imp_oracle': 100.0 * (L - L_oracle) / L.clamp_min(eps),
}

plot_features = {}
plot_features.update(model_features)
plot_features.update(diagnostic_features)
plot_features.update(oracle_diagnostics)

model_feature_df = make_feature_frame(model_features)
diagnostic_df = make_feature_frame(diagnostic_features)
oracle_diagnostic_df = make_feature_frame(oracle_diagnostics)
plot_feature_df = make_feature_frame(plot_features)

model_feature_names = sorted(model_features.keys())
diagnostic_feature_names = sorted(diagnostic_features.keys())
oracle_diagnostic_names = sorted(oracle_diagnostics.keys())
plot_feature_names = sorted(plot_features.keys())

feature_collections = {
    'model features': model_features,
    'target diagnostics': diagnostic_features,
    'oracle diagnostics': oracle_diagnostics,
    'all plot quantities': plot_features,
}

for collection_name, collection in feature_collections.items():
  if collection_name == 'all plot quantities':
    continue
  print(f'{collection_name}')
  display(list(sorted(collection.keys())))
  print("")


## Summary


In [ ]:
def _shape(x):
    if torch.is_tensor(x):
        return tuple(x.shape)
    if isinstance(x, np.ndarray):
        return tuple(x.shape)
    return None


def _numel(x):
    if torch.is_tensor(x):
        return x.numel()
    if isinstance(x, np.ndarray):
        return x.size
    return None


T, M, H = _shape(y)
K = _shape(yc)[2]
S = T * M

print(f"T: windows/individual - {T}")
print(f"M: individuals - {M}")
print(f"S: total windows - {S}")
print(f"K: neighbors/window - {K}")
print(f"H: horizon - {H}")
print("")

expected_shapes = {
    "y": [(T, M, H)],
    "yc": [(T, M, K, H)],
    "pred": [(T, M, H)],
    "pred_c": [(T, M, H)],
    "e": [(T, M, K, H)],
    "w": [(T, M, K)],
    "neighbor_weighted_mean": [(T, M, H)],
    "neighbor_unweighted_mean": [(T, M, H)],
    "weighted_yc": [(T, M, H)],
    "weighted_e": [(T, M, H)],
    "ell_vanilla": [(T, M, H)],
    "ell_context": [(T, M, H)],
}

for name, expected in expected_shapes.items():
    if name not in globals():
        print(f"{name}: missing")
        continue
    actual = _shape(globals()[name])
    if actual not in expected:
        print(f"{name}: expected one of {expected}, got {actual}")

feature_raw_expected_shapes = {
    "mu_x": [(T, M), (S,)],
    "sigma_x": [(T, M), (S,)],
    "mu_xc_mean": [(T, M), (S,)],
    "sigma_xc_mean": [(T, M), (S,)],
    "mu_xc_std": [(T, M), (S,)],
    "sigma_xc_std": [(T, M), (S,)],
    "same_individual_mean": [(T, M), (S,)],
    "delta_periods_mean": [(T, M), (S,)],
    "delta_periods_std": [(T, M), (S,)],
    "w_x_xc_entropy": [(T, M), (S,)],
    "w_x_xc_max": [(T, M), (S,)],
    "w_x_xc_std": [(T, M), (S,)],
    "loss_pred_pred_c": [(T, M), (S,)],
    "loss_pred_yc_mean": [(T, M), (S,)],
    "loss_neighbor_residual_mean": [(T, M), (S,)],
}

for name, expected in feature_raw_expected_shapes.items():
    if name not in feature_raw:
        print(f"feature_raw[{name!r}]: missing")
        continue
    actual = _shape(feature_raw[name])
    if actual not in expected:
        print(f"feature_raw[{name!r}]: expected one of {expected}, got {actual}")

for collection_name, collection in [
    ('model_features', model_features),
    ('diagnostic_features', diagnostic_features),
    ('oracle_diagnostics', oracle_diagnostics),
]:
    for name, value in collection.items():
        actual = _shape(value)
        if actual not in [(T, M), (S,)]:
            print(f"{collection_name}[{name!r}]: unexpected shape {actual}")

for forbidden in ['L_', 'Imp_', 'Rel_Imp_']:
    bad = [name for name in model_features if name.startswith(forbidden)]
    if bad:
        print(f"Target-dependent names found in model_features for prefix {forbidden}: {bad}")

print(f"model_feature_df shape = {model_feature_df.shape}")
print(f"diagnostic_df shape = {diagnostic_df.shape}")
print(f"oracle_diagnostic_df shape = {oracle_diagnostic_df.shape}")
print(f"plot_feature_df shape = {plot_feature_df.shape}")

total_scalar_targets = T * M * H
total_elements = 0

for name in expected_shapes:
    if name in globals():
        n = _numel(globals()[name])
        if n is not None:
            total_elements += n

for collection in [feature_raw, model_features, diagnostic_features, oracle_diagnostics]:
    for value in collection.values():
        n = _numel(value)
        if n is not None:
            total_elements += n

print(f"Total scalar targets = {total_scalar_targets:,}")
print(f"Total elements overall = {total_elements:,}")


#### Imported variables


- $y \in \mathbb{R}^{T \times M \times H}$ — `y`, target future.
- $y^c \in \mathbb{R}^{T \times M \times K \times H}$ — `yc`, retrieved neighbor futures.
- $p=f(x) \in \mathbb{R}^{T \times M \times H}$ — `pred`, vanilla prediction.
- $p^c=f(x,X_c) \in \mathbb{R}^{T \times M \times H}$ — `pred_c`, context-conditioned prediction.
- $e=y^c-p_k \in \mathbb{R}^{T \times M \times K \times H}$ — `e`, retrieved-neighbor vanilla residuals.
- $d \in \mathbb{R}^{T \times M \times K}$ — `distance`, raw retrieval distances.
- Query/neighbor time and user identifiers are loaded from `query_t`, `query_user_idx`, `neighbor_t`, and `neighbor_user_idx`.
- $w \in \mathbb{R}^{T \times M \times K}$ — `w`, weights computed in this notebook. By default, $w=\operatorname{softmax}(-d)$. Set `use_metadata_weights=True` to also apply a temporal penalty and same-user bonus.


#### Feature categories


The notebook keeps three separate dictionaries/dataframes.

1. `model_features` / `model_feature_df`: inference-available quantities. These may be used by learned gating/oracle-approximation models. They may depend on $x$, $p$, $p^c$, $y_k^c$, $e_k$, and $w_k$, but never on the current target $y$.
2. `diagnostic_features` / `diagnostic_df`: target-dependent losses and improvements. These are only for evaluation and plots.
3. `oracle_diagnostics` / `oracle_diagnostic_df`: target-dependent oracle upper bounds, also only for evaluation and plots.


#### Model features available at inference


Imported or derived model features include:

- $\mu_x$ — `mu_x`, input mean.
- $\sigma_x$ — `sigma_x`, input scale.
- $\overline{\mu_{x_c}}$ — `mu_xc_mean`, mean neighbor input mean.
- $\overline{\sigma_{x_c}}$ — `sigma_xc_mean`, mean neighbor input scale.
- $\operatorname{std}(\mu_{x_c})$ — `mu_xc_std`, dispersion of neighbor input means.
- $\operatorname{std}(\sigma_{x_c})$ — `sigma_xc_std`, dispersion of neighbor input scales.
- $K^{-1}\sum_k \mathbf{1}[u_k=u]$ — `same_individual_mean`, fraction of neighbors from the same individual.
- $K^{-1}\sum_k \Delta\tau_k$ — `delta_periods_mean`, mean neighbor age measured in periods.
- $\operatorname{std}_k(\Delta\tau_k)$ — `delta_periods_std`, dispersion of neighbor ages.
- $-\sum_k w_k\log w_k$ — `w_x_xc_entropy`, retrieval-weight entropy.
- $\max_k w_k$ — `w_x_xc_max`, largest retrieval weight.
- $\operatorname{std}_{k\le K}(w_k)$ — `w_x_xc_std`, dispersion of retrieval weights.
- $D_{p,p^c}=H^{-1}\sum_h(p_h-p^c_h)^2$ — `loss_pred_pred_c`, prediction disagreement.
- $D_{p,y^c}=K^{-1}\sum_k H^{-1}\sum_h(p_h-y^c_{k,h})^2$ — `loss_pred_yc_mean`, mean prediction-neighbor loss.
- $D_e=K^{-1}\sum_k H^{-1}\sum_h e_{k,h}^2$ — `loss_neighbor_residual_mean`, mean neighbor vanilla residual loss.
- `loss_pred_neighbor_weighted_mean`, `loss_pred_neighbor_unweighted_mean`, `loss_yc_pairwise`, `weighted_e_norm` are additional inference-available diagnostics derived from retrieved quantities only.


#### Computed prediction quantities available at inference


- $\overline{y}^c_w=\sum_k w_k y^c_k$ — `neighbor_weighted_mean`, weighted neighbor future.
- $\overline{y}^c=K^{-1}\sum_k y^c_k$ — `neighbor_unweighted_mean`, unweighted neighbor future.
- $\overline e=\sum_k w_k e_k$ — `weighted_e`, weighted vanilla residual.
- $\Delta p=p^c-p$ — `pred_c - pred`, context correction.


#### Target-dependent losses and diagnostics


For each window and horizon, define
\[
\ell_h(q,y)=(q_h-y_h)^2.
\]
The window-level loss is
\[
L(q,y)=\frac{1}{H}\sum_{h=1}^H \ell_h(q,y).
\]
In horizon plots, curves aggregate $\ell_h$-based quantities over all windows for each fixed horizon. In scalar summaries, metrics aggregate $L$ or all $\ell_h$ values over all windows.

`diagnostic_features` contains target-dependent quantities such as:

- `L_vanilla`: $L(p,y)$.
- `L_context`: $L(p^c,y)$.
- `L_neighbor_weighted_mean`: $L(\overline{y}^c_w,y)$.
- `L_neighbor_unweighted_mean`: $L(\overline{y}^c,y)$.
- `L_pred_plus_weighted_e`: $L(p+\overline e,y)$.
- `Imp_*`: $L(p,y)-L(q,y)$.
- `Rel_Imp_*`: $100\,[L(p,y)-L(q,y)]/L(p,y)$.


#### Oracle diagnostics involving $y$


These are diagnostic upper bounds, not deployable predictors, because they choose using the current target $y$.

- `L_oracle`: $\min_q L(q,y)$ over the exploratory candidates listed above.
- `Imp_oracle`: $L(p,y)-L_{\mathrm{oracle}}$.
- `Rel_Imp_oracle`: $100\,[L(p,y)-L_{\mathrm{oracle}}]/L(p,y)$.
- In the oracle horizon widget, including `y` among the oracle choices sets $\ell_h(y,y)=0$, so the relative improvement is $100\%$ at every horizon.


# Plots


## Histograms


In [ ]:
if 'symlog_np' not in globals():
    def symlog_np(x):
        return np.sign(x) * np.log1p(np.abs(x))


def feature_options_for_collection(collection_name):
    return sorted(feature_collections[collection_name].keys())


hist_collection_dropdown = widgets.Dropdown(
    options=list(feature_collections.keys()),
    value='all plot quantities',
    description='source:',
    layout=widgets.Layout(width='360px'),
)

hist_feature_dropdown = widgets.Dropdown(
    options=feature_options_for_collection(hist_collection_dropdown.value),
    description='feature:',
    layout=widgets.Layout(width='420px'),
)

if 'L_vanilla' in hist_feature_dropdown.options:
    hist_feature_dropdown.value = 'L_vanilla'

hist_symlog_toggle = widgets.ToggleButton(
    value=True,
    description='symlog on',
    tooltip='Toggle symlog transform on the histogram feature',
    icon='check',
)

hist_per_window_toggle = widgets.ToggleButton(
    value=False,
    description='per-user avg',
    tooltip='Toggle per-window points. Off = one point per user.',
    icon='users',
)

hist_inorm_toggle = widgets.ToggleButton(
    value=False,
    description='instance norm off',
    tooltip='Toggle instance-normalized feature values',
    icon='times',
)

hist_bins_slider = widgets.IntSlider(
    value=50,
    min=10,
    max=200,
    step=5,
    description='bins:',
    layout=widgets.Layout(width='320px'),
)

hist_output = widgets.Output()


def draw_hist(*_):
    with hist_output:
        clear_output(wait=True)

        source = feature_collections[hist_collection_dropdown.value]
        name = hist_feature_dropdown.value
        values = source[name].detach().cpu().numpy()

        if hist_per_window_toggle.value or values.ndim < 2:
            x = values.ravel()
        else:
            x = np.nanmean(values, axis=0).ravel()

        x = x[np.isfinite(x)]
        if hist_symlog_toggle.value:
            x = symlog_np(x)

        plt.figure(figsize=(8, 4))
        plt.hist(x, bins=hist_bins_slider.value)
        plt.title(f'{hist_collection_dropdown.value}: {name}')
        plt.xlabel('symlog(value)' if hist_symlog_toggle.value else 'value')
        plt.ylabel('count')
        plt.tight_layout()
        plt.show()

        if len(x):
            print(pd.Series(x).describe())


def update_hist_feature_options(*_):
    options = feature_options_for_collection(hist_collection_dropdown.value)
    hist_feature_dropdown.options = options
    preferred = 'L_vanilla' if hist_collection_dropdown.value != 'model features' else 'loss_pred_pred_c'
    hist_feature_dropdown.value = preferred if preferred in options else options[0]
    draw_hist()


def update_hist_labels(*_):
    hist_symlog_toggle.description = 'symlog on' if hist_symlog_toggle.value else 'symlog off'
    hist_per_window_toggle.description = 'per-window' if hist_per_window_toggle.value else 'per-user avg'
    draw_hist()


hist_collection_dropdown.observe(update_hist_feature_options, names='value')
for widget in [hist_feature_dropdown, hist_bins_slider]:
    widget.observe(draw_hist, names='value')
for widget in [hist_symlog_toggle, hist_per_window_toggle]:
    widget.observe(update_hist_labels, names='value')

display(widgets.VBox([
    widgets.HBox([hist_collection_dropdown, hist_feature_dropdown]),
    widgets.HBox([hist_bins_slider, hist_symlog_toggle, hist_per_window_toggle]),
    hist_output,
]))

draw_hist()


## Scatterplots


In [ ]:
def symlog_np(x):
    return np.sign(x) * np.log1p(np.abs(x))


def current_xy(x_name, y_name, x_collection, y_collection, symlog_x=True, symlog_y=True, per_user=True):
    collections = feature_collections
    x_source = collections[x_collection]
    y_source = collections[y_collection]
    x = x_source[x_name].detach().cpu().numpy()
    y_ = y_source[y_name].detach().cpu().numpy()

    if per_user and x.ndim >= 2 and y_.ndim >= 2:
        x = np.nanmean(x, axis=0).ravel()
        y_ = np.nanmean(y_, axis=0).ravel()
    else:
        x = x.ravel()
        y_ = y_.ravel()

    mask = np.isfinite(x) & np.isfinite(y_)
    x = x[mask]
    y_ = y_[mask]

    if symlog_x:
        x = symlog_np(x)
    if symlog_y:
        y_ = symlog_np(y_)

    return x, y_


def set_dropdown_options(dropdown, options, preferred=None):
    dropdown.options = options
    if preferred in options:
        dropdown.value = preferred
    elif options:
        dropdown.value = options[0]


x_collection_dropdown = widgets.Dropdown(
    options=list(feature_collections.keys()),
    value='model features',
    description='x source:',
    layout=widgets.Layout(width='320px'),
)

y_collection_dropdown = widgets.Dropdown(
    options=list(feature_collections.keys()),
    value='target diagnostics',
    description='y source:',
    layout=widgets.Layout(width='320px'),
)

x_dropdown = widgets.Dropdown(
    options=feature_options_for_collection(x_collection_dropdown.value),
    description='x:',
    layout=widgets.Layout(width='360px'),
)
set_dropdown_options(x_dropdown, list(x_dropdown.options), preferred='loss_pred_pred_c')

y_dropdown = widgets.Dropdown(
    options=feature_options_for_collection(y_collection_dropdown.value),
    description='y:',
    layout=widgets.Layout(width='360px'),
)
set_dropdown_options(y_dropdown, list(y_dropdown.options), preferred='Rel_Imp_context')

symlog_y_toggle = widgets.ToggleButton(value=True, description='symlog y on', icon='check')
symlog_x_toggle = widgets.ToggleButton(value=True, description='symlog x on', icon='check')
scatter_per_window_toggle = widgets.ToggleButton(value=False, description='per-user avg', icon='users')

scatter_alpha_slider = widgets.FloatSlider(
    value=0.35,
    min=0.02,
    max=1.0,
    step=0.02,
    description='alpha:',
    layout=widgets.Layout(width='320px'),
)

scatter_output = widgets.Output()


def draw_scatter(*_):
    with scatter_output:
        clear_output(wait=True)

        x, y_ = current_xy(
            x_dropdown.value,
            y_dropdown.value,
            x_collection_dropdown.value,
            y_collection_dropdown.value,
            symlog_x=symlog_x_toggle.value,
            symlog_y=symlog_y_toggle.value,
            per_user=not scatter_per_window_toggle.value,
        )

        plt.figure(figsize=(6, 6))
        plt.scatter(x, y_, s=12, alpha=scatter_alpha_slider.value)
        plt.xlabel(f'{x_collection_dropdown.value}: {x_dropdown.value}')
        plt.ylabel(f'{y_collection_dropdown.value}: {y_dropdown.value}')
        plt.title(f'{y_dropdown.value} vs {x_dropdown.value}')
        plt.tight_layout()
        plt.show()

        if len(x) > 1:
            corr = np.corrcoef(x, y_)[0, 1]
            print(f'n={len(x)}, corr={corr:.4f}')


def update_x_options(*_):
    set_dropdown_options(
        x_dropdown,
        feature_options_for_collection(x_collection_dropdown.value),
        preferred='loss_pred_pred_c',
    )
    draw_scatter()


def update_y_options(*_):
    set_dropdown_options(
        y_dropdown,
        feature_options_for_collection(y_collection_dropdown.value),
        preferred='Rel_Imp_context',
    )
    draw_scatter()


def update_scatter_labels(*_):
    symlog_y_toggle.description = 'symlog y on' if symlog_y_toggle.value else 'symlog y off'
    symlog_x_toggle.description = 'symlog x on' if symlog_x_toggle.value else 'symlog x off'
    scatter_per_window_toggle.description = 'per-window' if scatter_per_window_toggle.value else 'per-user avg'
    draw_scatter()


x_collection_dropdown.observe(update_x_options, names='value')
y_collection_dropdown.observe(update_y_options, names='value')
for widget in [x_dropdown, y_dropdown, scatter_alpha_slider]:
    widget.observe(draw_scatter, names='value')
for widget in [symlog_y_toggle, symlog_x_toggle, scatter_per_window_toggle]:
    widget.observe(update_scatter_labels, names='value')

display(widgets.VBox([
    widgets.HBox([x_collection_dropdown, x_dropdown]),
    widgets.HBox([y_collection_dropdown, y_dropdown]),
    widgets.HBox([symlog_x_toggle, symlog_y_toggle, scatter_per_window_toggle]),
    scatter_alpha_slider,
    scatter_output,
]))

draw_scatter()


## Horizon errors


In [ ]:
def to_numpy_horizon(x):
    if torch.is_tensor(x):
        x = x.detach().cpu().numpy()
    arr = np.asarray(x)
    return arr.reshape(-1, arr.shape[-1])


Y_all = to_numpy_horizon(y)
pred_all = to_numpy_horizon(pred)
pred_c_all = to_numpy_horizon(pred_c)
neighbor_weighted_mean_all = to_numpy_horizon(neighbor_weighted_mean)
neighbor_unweighted_mean_all = to_numpy_horizon(neighbor_unweighted_mean)
weighted_e_all = to_numpy_horizon(weighted_e)

eps_np = 1e-8
H_np = Y_all.shape[-1]

exploratory_horizon_predictions = {
    'vanilla': pred_all,
    'neighbor_conditioned': pred_c_all,
    'neighbor_weighted_mean': neighbor_weighted_mean_all,
    'neighbor_unweighted_mean': neighbor_unweighted_mean_all,
    'pred + weighted_e': pred_all + weighted_e_all,
}


def aggregate_over_windows(values, mode):
    if mode == 'median':
        return np.nanmedian(values, axis=0)
    return np.nanmean(values, axis=0)


def squared_error_horizon_np(y_true, y_pred):
    return (y_true - y_pred) ** 2


def relative_prediction_error_np(y_true, y_pred):
    return 100.0 * (y_pred - y_true) / np.maximum(np.abs(y_true), eps_np)


def relative_improvement_horizon_np(y_true, pred_ref, y_pred, oracle=False):
    base_lh = squared_error_horizon_np(y_true, pred_ref)
    model_lh = squared_error_horizon_np(y_true, y_pred)
    if oracle:
        model_lh = np.minimum(base_lh, model_lh)
    return 100.0 * (base_lh - model_lh) / np.maximum(base_lh, eps_np)


def oracle_loss_horizon_np(y_true, pred_ref, predictions, selected, oracle_mode):
    """
    Oracle over vanilla + selected candidates.

    horizon-wise:
        choose the best candidate independently for each (window, horizon).

    scalar:
        choose one candidate per window using L = mean_h ell_h,
        then use the chosen candidate's horizon-wise losses for plotting.
    """
    base_lh = squared_error_horizon_np(y_true, pred_ref)  # (N, H)
    candidate_losses_h = [base_lh]

    for name in selected:
        if name == 'vanilla':
            continue
        if name == 'y':
            candidate_losses_h.append(np.zeros_like(base_lh))
        else:
            candidate_losses_h.append(squared_error_horizon_np(y_true, predictions[name]))

    losses_h = np.stack(candidate_losses_h, axis=0)  # (C, N, H)

    if oracle_mode == 'horizon-wise':
        return np.nanmin(losses_h, axis=0)  # (N, H)

    if oracle_mode == 'scalar':
        losses_scalar = np.nanmean(losses_h, axis=2)  # (C, N)
        best_idx = np.nanargmin(losses_scalar, axis=0)  # (N,)
        return losses_h[best_idx, np.arange(losses_h.shape[1]), :]  # (N, H)

    raise ValueError(f'Unknown oracle_mode: {oracle_mode}')


def oracle_relative_improvement_horizon_np(y_true, pred_ref, predictions, selected, oracle_mode):
    base_lh = squared_error_horizon_np(y_true, pred_ref)
    oracle_lh = oracle_loss_horizon_np(
        y_true=y_true,
        pred_ref=pred_ref,
        predictions=predictions,
        selected=selected,
        oracle_mode=oracle_mode,
    )
    return 100.0 * (base_lh - oracle_lh) / np.maximum(base_lh, eps_np)


def display_horizon_diagnostic_widgets(predictions, y_true, pred_ref, title):
    if not predictions:
        print('No predictions available.')
        return

    names = list(predictions.keys())
    non_y_names = [name for name in names if name != 'y'] or names

    pred_dropdown = widgets.Dropdown(
        options=non_y_names,
        value='neighbor_conditioned' if 'neighbor_conditioned' in non_y_names else non_y_names[0],
        description='prediction:',
        layout=widgets.Layout(width='520px'),
    )
    relimp_dropdown = widgets.Dropdown(
        options=non_y_names,
        value='neighbor_conditioned' if 'neighbor_conditioned' in non_y_names else non_y_names[0],
        description='model:',
        layout=widgets.Layout(width='520px'),
    )
    oracle_choices = widgets.SelectMultiple(
        options=names + (['y'] if 'y' not in names else []),
        value=tuple([name for name in ['neighbor_conditioned'] if name in names] or [non_y_names[0]]),
        description='choices:',
        layout=widgets.Layout(width='520px', height='140px'),
    )

    aggregate_dropdown = widgets.Dropdown(
        options=['mean', 'median'],
        value='median',
        description='aggregate windows:',
        layout=widgets.Layout(width='260px'),
    )
    oracle_mode_dropdown = widgets.Dropdown(
        options=['horizon-wise', 'scalar'],
        value='horizon-wise',
        description='oracle mode:',
        layout=widgets.Layout(width='260px'),
    )
    symlog_toggle = widgets.ToggleButton(value=True, description='symlog on', icon='check')
    relimp_oracle_toggle = widgets.ToggleButton(value=False, description='oracle off', icon='times')

    pred_output = widgets.Output()
    relimp_output = widgets.Output()
    oracle_output = widgets.Output()

    def maybe_symlog(values):
        return symlog_np(values) if symlog_toggle.value else values

    def ylabel(label):
        return f'symlog({label})' if symlog_toggle.value else label

    def draw_prediction_relative(*_):
        with pred_output:
            clear_output(wait=True)
            name = pred_dropdown.value
            values = relative_prediction_error_np(y_true, predictions[name])
            curve = aggregate_over_windows(values, aggregate_dropdown.value)

            plt.figure(figsize=(9, 4))
            plt.plot(np.arange(1, y_true.shape[1] + 1), maybe_symlog(curve), marker='o')
            plt.axhline(0.0, linewidth=1)
            plt.xlabel('horizon')
            plt.ylabel(ylabel(r'100 $(q_h-y_h)/|y_h|$'))
            plt.title(f'{title}: relative prediction error — {name}')
            plt.tight_layout()
            plt.show()

    def draw_relative_improvement(*_):
        with relimp_output:
            clear_output(wait=True)
            name = relimp_dropdown.value
            values = relative_improvement_horizon_np(
                y_true,
                pred_ref,
                predictions[name],
                oracle=relimp_oracle_toggle.value,
            )
            curve = aggregate_over_windows(values, aggregate_dropdown.value)
            title_suffix = (
                f'horizon-wise oracle(vanilla, {name})'
                if relimp_oracle_toggle.value
                else name
            )

            plt.figure(figsize=(9, 4))
            plt.plot(np.arange(1, y_true.shape[1] + 1), maybe_symlog(curve), marker='o')
            plt.axhline(0.0, linewidth=1)
            plt.xlabel('horizon')
            plt.ylabel(ylabel(r'100 $(\ell_p-\ell_g)/\ell_p$'))
            plt.title(f'{title}: relative improvement — {title_suffix}')
            plt.tight_layout()
            plt.show()

    def draw_oracle(*_):
        with oracle_output:
            clear_output(wait=True)
            selected = list(oracle_choices.value)
            if not selected:
                print('Select at least one oracle candidate.')
                return

            oracle_values = oracle_relative_improvement_horizon_np(
                y_true=y_true,
                pred_ref=pred_ref,
                predictions=predictions,
                selected=selected,
                oracle_mode=oracle_mode_dropdown.value,
            )
            oracle_curve = aggregate_over_windows(oracle_values, aggregate_dropdown.value)

            oracle_color = 'red' if 'y' in selected else None
            plt.plot(
                np.arange(1, y_true.shape[1] + 1),
                maybe_symlog(oracle_curve),
                marker='o',
                linewidth=2.5,
                color=oracle_color,
                label=f'oracle over vanilla + selected ({oracle_mode_dropdown.value})',
            )

            plt.axhline(0.0, linewidth=1)
            plt.xlabel('horizon')
            plt.ylabel(ylabel(r'100 $(\ell_p-\ell_{\mathrm{oracle}})/\ell_p$'))
            plt.title(f'{title}: oracle relative improvement')
            plt.legend(fontsize=8)
            plt.tight_layout()
            plt.show()

    def draw_all(*_):
        draw_prediction_relative()
        draw_relative_improvement()
        draw_oracle()

    def update_labels(*_):
        symlog_toggle.description = 'symlog on' if symlog_toggle.value else 'symlog off'
        symlog_toggle.icon = 'check' if symlog_toggle.value else 'times'
        relimp_oracle_toggle.description = 'oracle on' if relimp_oracle_toggle.value else 'oracle off'
        relimp_oracle_toggle.icon = 'check' if relimp_oracle_toggle.value else 'times'
        draw_all()

    pred_dropdown.observe(draw_prediction_relative, names='value')
    relimp_dropdown.observe(draw_relative_improvement, names='value')
    oracle_choices.observe(draw_oracle, names='value')
    aggregate_dropdown.observe(draw_all, names='value')
    oracle_mode_dropdown.observe(draw_oracle, names='value')
    symlog_toggle.observe(update_labels, names='value')
    relimp_oracle_toggle.observe(update_labels, names='value')

    controls = widgets.VBox([
        widgets.HTML('<b>Aggregation is over all windows for each fixed horizon.</b>'),
        widgets.HBox([aggregate_dropdown, symlog_toggle]),
        widgets.HTML('<b>1. Relative prediction error versus y</b>'),
        pred_dropdown,
        pred_output,
        widgets.HTML('<b>2. Relative improvement versus vanilla</b>'),
        widgets.HBox([relimp_dropdown, relimp_oracle_toggle]),
        relimp_output,
        widgets.HTML('<b>3. Oracle over vanilla + selected choices</b>'),
        widgets.HBox([oracle_mode_dropdown]),
        oracle_choices,
        oracle_output,
    ])
    display(controls)
    draw_all()


display_horizon_diagnostic_widgets(
    exploratory_horizon_predictions,
    Y_all,
    pred_all,
    'Exploratory horizon diagnostics',
)


# Models


## Model descriptions


We compare four neighbor-augmented baselines against the vanilla pretrained forecast $\hat y=f(x)$. Let $Y=(y_k)_{k=1}^K$ and $\hat y^c=f(x,\mathcal N(x))$. By default, $w_k=\exp(-d_k)/\sum_j\exp(-d_j)$. Setting `use_metadata_weights=True` also penalizes temporal distance and rewards neighbors from the same user.

1. **Weighted mixture of horizons**:
   $\hat y_{mix_0}=(1-\lambda)\hat y+\lambda\sum_{k=1}^K w_k y_k$.
2. **Learned mixture of horizons**:
   $\hat y_{mix_1}=\hat y+a_0\hat y+\sum_{k=1}^K a_k y_k$.
3. **Full horizon-wise mixture**:
   $\hat y_{mix_2,h}=\hat y_h+a_{0,h}\hat y_h+\sum_{k=1}^K a_{k,h}y_{k,h}+b_h\hat y^c_h$.
4. **Gated context-conditioned forecast**:
   Scalar and horizon-wise gates use inference features derived from $x$, $\mathcal N(x)$, $\hat y$, and $\hat y^c$ to choose between $\hat y$ and $\hat y^c$.

The first two mixtures share their coefficients across horizons. In the full mixture, all coefficients $(a_{0,h},a_{k,h},b_h)$ are horizon-specific. The linear models are regularized fits to residual labels $y-\hat y$, so zero correction recovers the vanilla forecast. The context-conditioned candidate used by the learned gates remains available as `context_conditioned`; scalar gates and optional horizon-wise gates are fitted in the oracle section.

## Data splits and model utilities


In [ ]:
baseline_l2 = 1e-3
random_state = 0

# The last 15% of dates from the former training payload are reserved for
# fitting the oracle. The untouched evaluation payload is the test split.
oracle_train_fraction = 0.15


def split_prediction_arrays(split):
    payload = prediction_payloads[split]
    split_w = split_weights[split]
    out = {
        'Y': get_tensor(payload, split, 'Y_values').float().reshape(-1, get_tensor(payload, split, 'Y_values').shape[-1]).numpy(),
        'pred': get_tensor(payload, split, 'preds').float().reshape(-1, get_tensor(payload, split, 'preds').shape[-1]).numpy(),
        'pred_c': get_tensor(payload, split, 'preds_context').float().reshape(-1, get_tensor(payload, split, 'preds_context').shape[-1]).numpy(),
        'yc': get_tensor(payload, split, 'Yc_values').float().reshape(-1, *get_tensor(payload, split, 'Yc_values').shape[2:]).numpy(),
        'w': split_w.reshape(-1, split_w.shape[-1]).numpy(),
        'distance': get_tensor(payload, split, 'distance_x_xc').float().reshape(-1, get_tensor(payload, split, 'distance_x_xc').shape[-1]).numpy(),
        'query_t': get_tensor(payload, split, 'query_t').long().reshape(-1).numpy(),
        'query_user_idx': get_tensor(payload, split, 'query_user_idx').long().reshape(-1).numpy(),
        'neighbor_t': get_tensor(payload, split, 'neighbor_t').long().reshape(-1, get_tensor(payload, split, 'neighbor_t').shape[-1]).numpy(),
        'neighbor_user_idx': get_tensor(payload, split, 'neighbor_user_idx').long().reshape(-1, get_tensor(payload, split, 'neighbor_user_idx').shape[-1]).numpy(),
    }
    retrieval_period = max(int(get_tensor(payload, split, 'retrieval_period').item()), 1)
    out['delta_periods'] = np.maximum(
        out['query_t'][:, None] - out['neighbor_t'],
        0,
    ) / retrieval_period
    out['same_individual'] = (
        out['query_user_idx'][:, None] == out['neighbor_user_idx']
    ).astype(np.float32)
    return out


def split_feature_frame(split):
    payload = features_payloads[split]
    cols = {}
    for key, value in payload.items():
        if not torch.is_tensor(value):
            continue
        flat = value.float().reshape(-1).numpy()
        expected = get_tensor(prediction_payloads[split], split, 'Y_values').shape[0] * get_tensor(
            prediction_payloads[split], split, 'Y_values'
        ).shape[1]
        if len(flat) == expected:
            cols[key.removeprefix(f'{split}_')] = flat
    return pd.DataFrame(cols)


def pairwise_yc_loss_np(yc):
    if yc.shape[1] <= 1:
        return np.zeros(yc.shape[0], dtype=yc.dtype)
    diff = yc[:, :, None, :] - yc[:, None, :, :]
    return np.mean(diff ** 2, axis=(1, 2, 3))


def add_inference_feature_columns(frame, arrays):
    w = arrays['w']
    pred = arrays['pred']
    pred_c = arrays['pred_c']
    yc = arrays['yc']
    delta_periods = arrays['delta_periods']
    same_individual = arrays['same_individual']
    neighbor_weighted = (w[:, :, None] * yc).sum(axis=1)
    neighbor_unweighted = yc.mean(axis=1)

    frame['w_x_xc_entropy'] = -(w * np.log(np.maximum(w, 1e-8))).sum(axis=1)
    frame['w_x_xc_max'] = w.max(axis=1)
    frame['w_x_xc_std'] = w.std(axis=1)
    frame['same_individual_mean'] = same_individual.mean(axis=1)
    frame['delta_periods_mean'] = delta_periods.mean(axis=1)
    frame['delta_periods_std'] = delta_periods.std(axis=1)
    frame['loss_pred_pred_c'] = np.mean((pred - pred_c) ** 2, axis=1)
    frame['loss_pred_neighbor_weighted_mean'] = np.mean((pred - neighbor_weighted) ** 2, axis=1)
    frame['loss_pred_neighbor_unweighted_mean'] = np.mean((pred - neighbor_unweighted) ** 2, axis=1)
    frame['loss_yc_pairwise'] = pairwise_yc_loss_np(yc)

    if 'mu_x' in frame.columns and 'mu_xc_mean' in frame.columns:
        frame['mu_x_minus_mu_xc_mean'] = frame['mu_x'] - frame['mu_xc_mean']
    if 'sigma_x' in frame.columns and 'sigma_xc_mean' in frame.columns:
        frame['sigma_x_minus_sigma_xc_mean'] = frame['sigma_x'] - frame['sigma_xc_mean']
    return frame


def valid_mask_from_arrays(arrays, features):
    values = [
        arrays['Y'], arrays['pred'], arrays['pred_c'], arrays['yc'], arrays['w'],
        features.to_numpy(),
    ]
    mask = np.ones(len(arrays['Y']), dtype=bool)
    for value in values:
        finite = np.isfinite(value)
        if finite.ndim > 1:
            finite = finite.reshape(finite.shape[0], -1).all(axis=1)
        mask &= finite
    return mask


def subset_arrays(arrays, mask):
    return {
        name: value[mask]
        for name, value in arrays.items()
        if isinstance(value, np.ndarray) and value.shape[0] == len(mask)
    }


source_train_arrays = split_prediction_arrays('train')
source_test_arrays = split_prediction_arrays('eval')
source_train_features = add_inference_feature_columns(
    split_feature_frame('train'), source_train_arrays
)
source_test_features = add_inference_feature_columns(
    split_feature_frame('eval'), source_test_arrays
)

valid_train = valid_mask_from_arrays(source_train_arrays, source_train_features)
valid_test = valid_mask_from_arrays(source_test_arrays, source_test_features)
source_train_arrays = subset_arrays(source_train_arrays, valid_train)
source_test_arrays = subset_arrays(source_test_arrays, valid_test)
source_train_features = source_train_features.loc[valid_train].reset_index(drop=True)
source_test_features = source_test_features.loc[valid_test].reset_index(drop=True)

train_dates = np.sort(np.unique(source_train_arrays['query_t']))
if len(train_dates) < 2:
    raise ValueError('At least two distinct training query dates are required.')
n_oracle_dates = max(1, int(np.ceil(len(train_dates) * oracle_train_fraction)))
n_oracle_dates = min(n_oracle_dates, len(train_dates) - 1)
oracle_start_t = train_dates[-n_oracle_dates]

baseline_train_mask = source_train_arrays['query_t'] < oracle_start_t
oracle_train_mask = ~baseline_train_mask

train_arrays = subset_arrays(source_train_arrays, baseline_train_mask)
oracle_train_arrays = subset_arrays(source_train_arrays, oracle_train_mask)
test_arrays = source_test_arrays

train_features_df = source_train_features.loc[baseline_train_mask].reset_index(drop=True)
oracle_train_features_df = source_train_features.loc[oracle_train_mask].reset_index(drop=True)
test_features_df = source_test_features

split_arrays = {
    'train': train_arrays,
    'oracle_train': oracle_train_arrays,
    'test': test_arrays,
}
split_features = {
    'train': train_features_df,
    'oracle_train': oracle_train_features_df,
    'test': test_features_df,
}

H = train_arrays['Y'].shape[1]
K = train_arrays['w'].shape[1]

for split_name, arrays in split_arrays.items():
    dates = arrays['query_t']
    print(
        f"{split_name}: {len(arrays['Y']):,} windows, "
        f"dates {int(dates.min())} -> {int(dates.max())}"
    )


base_model_formula_map = {
    'vanilla': r'$\hat y$',
    'mix_0_weighted': r'$(1-\lambda)\hat y+\lambda\sum_k w_ky_k$',
    'mix_1_learned': r'$\hat y+a_0\hat y+\sum_k a_ky_k$',
    'mix_2_full_horizon': r'$\hat y_h+a_{0,h}\hat y_h+\sum_k a_{k,h}y_{k,h}+b_h\hat y^c_h$',
    'context_conditioned': r'$\hat y^c$',
    'gated_context': r'$\sigma(x,\mathcal N(x),\hat y,\hat y^c)$',
}


def fit_linear_no_intercept(X_train, y_train, l2=baseline_l2):
    X = np.asarray(X_train, dtype=np.float64)
    y = np.asarray(y_train, dtype=np.float64)
    mask = np.isfinite(X).all(axis=1) & np.isfinite(y)
    X, y = X[mask], y[mask]
    if X.shape[0] == 0:
        return np.zeros(X.shape[1], dtype=np.float64)
    A = (X.T @ X) / X.shape[0] + l2 * np.eye(X.shape[1])
    b = (X.T @ y) / X.shape[0]
    return np.linalg.solve(A, b)


def weighted_neighbor_horizon(arrays):
    return (arrays['w'][:, :, None] * arrays['yc']).sum(axis=1)


def fit_baseline_models(train, l2=baseline_l2):
    weighted = weighted_neighbor_horizon(train)

    # mix_0 = pred + lambda * (weighted_neighbors - pred), lambda in [0, 1].
    # The penalty is toward lambda=0, which recovers the vanilla forecast.
    mix_0_direction = weighted - train['pred']
    mix_0_target = train['Y'] - train['pred']

    numerator = np.mean(mix_0_direction * mix_0_target)
    denominator = np.mean(mix_0_direction ** 2) + l2
    mix_0_lambda = np.clip(
        numerator / max(denominator, 1e-12),
        0.0,
        1.0,
    )

    # mix_1 = pred + a_0 * pred + sum_k a_k * y_k
    mix_1_X = np.concatenate(
        [
            train['pred'].reshape(-1, 1),
            train['yc'].transpose(0, 2, 1).reshape(-1, K),
        ],
        axis=1,
    )
    mix_1_coef = fit_linear_no_intercept(
        mix_1_X,
        (train['Y'] - train['pred']).reshape(-1),
        l2=l2,
    )

    # mix_2 fits an independent regularized residual correction per horizon:
    # pred_h + a_{0,h} pred_h + sum_k a_{k,h} y_{k,h} + b_h pred_c_h.
    mix_2_coef = np.zeros((H, K + 2), dtype=np.float64)
    for h in range(H):
        X_h = np.concatenate(
            [
                train['pred'][:, h:h + 1],
                train['yc'][:, :, h],
                train['pred_c'][:, h:h + 1],
            ],
            axis=1,
        )
        mix_2_coef[h] = fit_linear_no_intercept(
            X_h,
            train['Y'][:, h] - train['pred'][:, h],
            l2=l2,
        )

    return {
        'mix_0_lambda': float(mix_0_lambda),
        'mix_1_a0': float(mix_1_coef[0]),
        'mix_1_neighbor_coef': mix_1_coef[1:],
        'mix_2_a0': mix_2_coef[:, 0],
        'mix_2_horizon_coef': mix_2_coef[:, 1:],
    }


def predict_baseline_models(arrays, artifacts):
    predictions = {
        'vanilla': arrays['pred'].copy(),
        'context_conditioned': arrays['pred_c'].copy(),
    }

    weighted = weighted_neighbor_horizon(arrays)
    lam = artifacts['mix_0_lambda']
    predictions['mix_0_weighted'] = (
        (1.0 - lam) * arrays['pred']
        + lam * weighted
    )

    neighbor_contribution = (
        arrays['yc'].transpose(0, 2, 1).reshape(-1, K)
        @ artifacts['mix_1_neighbor_coef']
    ).reshape(-1, H)
    predictions['mix_1_learned'] = (
        (1.0 + artifacts['mix_1_a0']) * arrays['pred']
        + neighbor_contribution
    )

    full = arrays['pred'].copy()
    for h in range(H):
        X_h = np.concatenate(
            [arrays['yc'][:, :, h], arrays['pred_c'][:, h:h + 1]],
            axis=1,
        )
        full[:, h] += (
            artifacts['mix_2_a0'][h] * arrays['pred'][:, h]
            + X_h @ artifacts['mix_2_horizon_coef'][h]
        )
    predictions['mix_2_full_horizon'] = full
    return predictions


## Baseline model training

This cell fits adapters on `train` and generates predictions for all three periods. It intentionally does not display evaluation results.


In [ ]:
model_artifacts = fit_baseline_models(train_arrays)
model_predictions_by_split = {
    split_name: predict_baseline_models(arrays, model_artifacts)
    for split_name, arrays in split_arrays.items()
}

print('Baseline models trained on the train split.')
print('Available models:', list(model_predictions_by_split['test']))

# Oracle learning


## Oracle description


The baseline adapters are fitted only on `train`. The scalar oracle is then fitted on the smaller, later `oracle_train` period and evaluated on any selected split. This prevents the oracle from learning labels produced by baseline models on the same observations used to fit those baselines.

The horizon-wise oracle is optional because it trains one classifier per forecast horizon rather than one classifier per candidate.


In [ ]:
def fit_binary_oracle_classifier(X_train, y_train):
    if len(np.unique(y_train)) < 2:
        return {'constant': float(y_train.mean())}
    clf = CatBoostClassifier(
        iterations=300,
        learning_rate=0.03,
        depth=4,
        loss_function='Logloss',
        eval_metric='Accuracy',
        auto_class_weights='Balanced',
        random_seed=random_state,
        verbose=False,
        allow_writing_files=False,
    )
    clf.fit(X_train, y_train)
    return {'classifier': clf}


def predict_binary_oracle(model, X):
    if 'constant' in model:
        return np.full(len(X), model['constant'], dtype=np.float64)
    return model['classifier'].predict_proba(X)[:, 1]


def candidate_names(predictions):
    return [name for name in predictions if name != 'vanilla']

## Scalar oracle training


In [ ]:
def fit_scalar_oracle_models():
    arrays = oracle_train_arrays
    features = oracle_train_features_df.to_numpy(dtype=np.float64)
    predictions = model_predictions_by_split['oracle_train']
    base_loss = np.mean((arrays['Y'] - arrays['pred']) ** 2, axis=1)

    models = {}
    for name in candidate_names(predictions):
        candidate_loss = np.mean((arrays['Y'] - predictions[name]) ** 2, axis=1)
        labels = (candidate_loss < base_loss).astype(int)
        models[name] = fit_binary_oracle_classifier(features, labels)
    return models


def predict_scalar_oracles(split_name, models):
    arrays = split_arrays[split_name]
    features = split_features[split_name].to_numpy(dtype=np.float64)
    candidates = model_predictions_by_split[split_name]
    predictions = {}
    for name, model in models.items():
        probability = predict_binary_oracle(model, features)
        oracle_name = f'oracle(scalar)_{name}'
        predictions[oracle_name] = np.where(
            probability[:, None] > 0.5,
            candidates[name],
            arrays['pred'],
        )
    return predictions


scalar_oracle_models = fit_scalar_oracle_models()
scalar_oracle_predictions_by_split = {
    split_name: predict_scalar_oracles(split_name, scalar_oracle_models)
    for split_name in split_arrays
}
print(f'Trained {len(scalar_oracle_models)} scalar oracle classifiers.')

## Optional horizon-wise oracle training


In [ ]:
train_horizon_oracles = False


def fit_horizon_oracle_models():
    arrays = oracle_train_arrays
    features = oracle_train_features_df.to_numpy(dtype=np.float64)
    predictions = model_predictions_by_split['oracle_train']
    base_loss = (arrays['Y'] - arrays['pred']) ** 2

    models = {}
    for name in candidate_names(predictions):
        candidate_loss = (arrays['Y'] - predictions[name]) ** 2
        models[name] = [
            fit_binary_oracle_classifier(
                features,
                (candidate_loss[:, h] < base_loss[:, h]).astype(int),
            )
            for h in range(H)
        ]
    return models


def predict_horizon_oracles(split_name, models):
    arrays = split_arrays[split_name]
    features = split_features[split_name].to_numpy(dtype=np.float64)
    candidates = model_predictions_by_split[split_name]
    predictions = {}
    for name, horizon_models in models.items():
        probability = np.column_stack([
            predict_binary_oracle(model, features)
            for model in horizon_models
        ])
        oracle_name = f'oracle(horizon)_{name}'
        predictions[oracle_name] = np.where(
            probability > 0.5,
            candidates[name],
            arrays['pred'],
        )
    return predictions


if train_horizon_oracles:
    horizon_oracle_models = fit_horizon_oracle_models()
    horizon_oracle_predictions_by_split = {
        split_name: predict_horizon_oracles(split_name, horizon_oracle_models)
        for split_name in split_arrays
    }
    print(f'Trained {len(horizon_oracle_models) * H} horizon classifiers.')
else:
    horizon_oracle_models = {}
    horizon_oracle_predictions_by_split = {
        split_name: {}
        for split_name in split_arrays
    }
    print('Horizon-wise oracle training is disabled.')

# Interactive metrics


In [ ]:
all_predictions_by_split = {}
for split_name in split_arrays:
    all_predictions_by_split[split_name] = {
        **model_predictions_by_split[split_name],
        **scalar_oracle_predictions_by_split[split_name],
        **horizon_oracle_predictions_by_split[split_name],
    }


def normalize_instance_values(values, features):
    mu = features['mu_x'].to_numpy(dtype=np.float64)[:, None]
    sigma = features['sigma_x'].to_numpy(dtype=np.float64)[:, None]
    return (np.asarray(values, dtype=np.float64) - mu) / np.maximum(sigma, 1e-8)


def point_loss(y_true, y_pred, metric):
    error = np.asarray(y_pred) - np.asarray(y_true)
    if metric == 'MSE':
        return error ** 2
    if metric == 'MAE':
        return np.abs(error)
    raise ValueError(metric)


def metric_table(split_name, normalized, metric, aggregation):
    arrays = split_arrays[split_name]
    features = split_features[split_name]
    predictions = all_predictions_by_split[split_name]

    y_true = arrays['Y']
    evaluated_predictions = predictions
    if normalized:
        y_true = normalize_instance_values(y_true, features)
        evaluated_predictions = {
            name: normalize_instance_values(prediction, features)
            for name, prediction in predictions.items()
        }

    comparison_metric = 'MSE' if metric == 'positive windows %' else metric
    vanilla_point_loss = point_loss(
        y_true,
        evaluated_predictions['vanilla'],
        comparison_metric,
    )
    vanilla_window_loss = vanilla_point_loss.mean(axis=1)
    vanilla_aggregate = vanilla_point_loss.mean()

    rows = []
    for name, prediction in evaluated_predictions.items():
        losses = point_loss(y_true, prediction, comparison_metric)
        window_loss = losses.mean(axis=1)

        if metric == 'positive windows %':
            value = 100.0 * np.mean(window_loss < vanilla_window_loss)
            unit = '%'
        elif aggregation == 'default':
            value = losses.mean()
            unit = metric
        elif aggregation == 'relative improvement':
            value = 100.0 * (
                vanilla_aggregate - losses.mean()
            ) / max(vanilla_aggregate, 1e-12)
            unit = '%'
        elif aggregation == 'average relative improvement':
            value = 100.0 * np.mean(
                (vanilla_window_loss - window_loss)
                / np.maximum(vanilla_window_loss, 1e-12)
            )
            unit = '%'
        elif aggregation == 'average horizon loss':
            value = losses.mean(axis=0).mean()
            unit = metric
        else:
            raise ValueError(aggregation)

        rows.append({
            'model': name,
            'value': float(value),
            'unit': unit,
        })

    frame = pd.DataFrame(rows)
    higher_is_better = (
        metric == 'positive windows %'
        or aggregation in {'relative improvement', 'average relative improvement'}
    )
    return frame.sort_values(
        'value',
        ascending=not higher_is_better,
    ).reset_index(drop=True)


split_dropdown = widgets.Dropdown(
    options=[
        ('training', 'train'),
        ('oracle training', 'oracle_train'),
        ('testing', 'test'),
    ],
    value='test',
    description='split:',
)
normalization_toggle = widgets.ToggleButtons(
    options=[('raw', False), ('normalized', True)],
    value=False,
    description='values:',
)
metric_dropdown = widgets.Dropdown(
    options=['MSE', 'MAE', 'positive windows %'],
    value='MSE',
    description='metric:',
)
aggregation_dropdown = widgets.Dropdown(
    options=[
        'default',
        'relative improvement',
        'average relative improvement',
        'average horizon loss',
    ],
    value='default',
    description='aggregation:',
    layout=widgets.Layout(width='360px'),
)
metrics_output = widgets.Output()


def draw_metric_table(*_):
    aggregation_dropdown.disabled = metric_dropdown.value == 'positive windows %'
    with metrics_output:
        clear_output(wait=True)
        frame = metric_table(
            split_name=split_dropdown.value,
            normalized=normalization_toggle.value,
            metric=metric_dropdown.value,
            aggregation=aggregation_dropdown.value,
        )
        split_label = {
            'train': 'training',
            'oracle_train': 'oracle training',
            'test': 'testing',
        }[split_dropdown.value]
        caption = (
            f"{split_label}; "
            f"{'instance-normalized' if normalization_toggle.value else 'raw'} values; "
            f"{metric_dropdown.value}"
        )
        if metric_dropdown.value != 'positive windows %':
            caption += f"; {aggregation_dropdown.value}"
        print(caption)
        display(frame.style.format({'value': '{:.4f}'}))


for control in [
    split_dropdown,
    normalization_toggle,
    metric_dropdown,
    aggregation_dropdown,
]:
    control.observe(draw_metric_table, names='value')

display(widgets.VBox([
    widgets.HBox([split_dropdown, normalization_toggle]),
    widgets.HBox([metric_dropdown, aggregation_dropdown]),
    metrics_output,
]))
draw_metric_table()